# Project 1: Grounded RAG Pipeline for Document Q&A
**Tech Stack**: Python, LangChain, ChromaDB, Sentence-Transformers, Gemini API


## 1. Environment Setup & Dependency Installation

In [22]:

pip install -q google-genai pypdf langchain-text-splitters sentence-transformers chromadb

## 2. Document Ingestion

In [6]:
from google.colab import files
print("upload a PDF document to test yout app")
uploaded=files.upload()
#get the uploaded file name automatically
pdf_filename=list(uploaded.keys())[0]
print(f"\nSuccessfully uploaded:{pdf_filename}")

upload a PDF document to test yout app


Saving CI_CD and Devops Notes.pdf to CI_CD and Devops Notes.pdf

Successfully uploaded:CI_CD and Devops Notes.pdf


## 3. Vector Database Indexing & Gemini Query Pipeline

In [18]:
import os
from google import genai
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

# 1. Initialize Gemini Client
GEMINI_API_KEY = "AQ.Ab8RN6LdqQ1_IXXY5aQUjMOiqZbpKEy8EVRHUmuAo1Ww3wGrWg"  # <-- Put your API key here
client = genai.Client(api_key=GEMINI_API_KEY)

# 2. Extract Text from the PDF
print("Reading PDF...")
loader = PyPDFLoader(pdf_filename)
documents = loader.load()

# 3. Chunk Text into Larger Pieces (Updated chunk_size=1000)
print("Splitting text into chunks...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)
chunk_texts = [doc.page_content for doc in chunks]

# 4. Generate Embeddings & Store in Vector Database
print("Generating vector embeddings and saving to ChromaDB...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(chunk_texts).tolist()

# Create or Get Local ChromaDB Collection
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="pdf_rag_updated")

# Add chunks and embeddings to the database
ids = [f"id_{i}" for i in range(len(chunk_texts))]
collection.add(documents=chunk_texts, embeddings=embeddings, ids=ids)
print("Database indexing complete!\n" + "="*50)

# 5. Question-Answering Function
def ask_pdf(question: str):
    # Convert query to embedding
    query_embedding = embedder.encode([question]).tolist()

    # Retrieve top 5 most relevant chunks (Updated n_results=5)
    results = collection.query(query_embeddings=query_embedding, n_results=5)
    retrieved_context = "\n---\n".join(results['documents'][0])

    # Construct Grounded Prompt
    prompt = f"""You are a helpful assistant. Answer the user's question using ONLY the provided context from the PDF document.
If the answer is not contained in the context, reply with "I cannot find the answer in the provided document."

Context:
{retrieved_context}

Question: {question}
Answer:"""

    # Generate Response using Gemini
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )
    return response.text

# --- Test Your App ---
# Type a specific question about your PDF here
user_question = "What is the main topic or key takeaway of this document?"
print(f"Question: {user_question}\n")
answer = ask_pdf(user_question)
print(f"Answer:\n{answer}")

Reading PDF...
Splitting text into chunks...
Generating vector embeddings and saving to ChromaDB...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Database indexing complete!
Question: What is the main topic or key takeaway of this document?

Answer:
Based on the provided document, the main topic is **DevOps**, specifically focusing on:

*   **The DevOps Lifecycle and Workflow:** Explaining the continuous cycle of Plan, Develop, Build, Test, Release, Deploy, Operate, Monitor, and Feedback, using real-time scenarios (like a Swiggy feature release).
*   **CI/CD (Continuous Integration / Continuous Delivery or Continuous Deployment):** Described as the "backbone of DevOps," detailing how developers integrate code and how it is automatically tested, built, and deployed to production.
*   **Daily Activities of a DevOps Engineer:** Including infrastructure management, monitoring system health (CPU, memory, disk, network, and application errors using tools like Grafana and Prometheus), and container management (using Docker and Kubernetes).


In [20]:
pip install -q gradio


## 4. Interactive Web Interface

In [21]:
import gradio as gr

def respond(message, history):
    # Uses your existing ask_pdf function
    response = ask_pdf(message)
    return response

# Launch a clean chat interface
demo = gr.ChatInterface(
    fn=respond,
    title="📄 AI Document Assistant (RAG Pipeline)",
    description="Ask questions about your uploaded PDF. Powered by ChromaDB vector search and Gemini.",
    examples=["What is the main topic of this document?", "Summarize the key key points."]
)

# share=True creates a public 72-hour link you can record in a video demo!
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2ab14768c275dbb5a4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2ab14768c275dbb5a4.gradio.live
